# 02 · Exploratory Data Analysis (rubric: 15 pts)
**Owner:** A (project lead) · **Phase 1** · Figures → `outputs/figures/`, tables → `outputs/tables/`

Rubric checklist (tick each in the report):
- [ ] sources, record counts, missing values, schema  (§1)
- [ ] token distribution, vocabulary size, sequence length  (§3)
- [ ] charts: word cloud, frequency plots, embedding map, similarity heatmap  (§3–5)
- [ ] label / topic distribution  (§2)
- [ ] linguistic features: n-grams, POS tags, entities  (§4)
- [ ] generated (synthetic) data: quality, diversity, bias, anomalies, edge cases  (§6)

In [1]:
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:  # Colab: clone the team repo once per session, install requirements, load API keys from Secrets
    REPO_URL = "https://github.com/sh1or1tama1/banking-support-agent.git"
    if not os.path.exists("/content/banking-support-agent"):
        subprocess.run(["git", "clone", REPO_URL, "/content/banking-support-agent"], check=True)
    os.chdir("/content/banking-support-agent")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    from google.colab import userdata
    for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
sys.path.insert(0, os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else "."))

import pandas as pd
from IPython.display import display
from src import config as C
from src.data.load import load_banking77
from src.eda import eda
from src.embeddings import Embedder

df = pd.read_parquet(C.SPLITS_FILE)          # frozen split from notebook 01
b77 = load_banking77()
pd.set_option("display.max_colwidth", 120)

def save(t, name):
    t.to_csv(C.TABLES / f"{name}.csv", index=False); return t

## 1. Sources, schema, missing values, quality
| Dataset | Source | Role | Rows |
|---|---|---|---|
| Bitext Retail Banking (synthetic, CDLA-Sharing-1.0) | huggingface.co/datasets/bitext/Bitext-retail-banking-llm-chatbot-training-dataset | train/val/test, RAG knowledge base | see below |
| BANKING77 (real customer queries, CC-BY-4.0) | github.com/PolyAI-LDN/task-specific-datasets | external test (real wording), unsupported-intent queries | 13,083 |

In [2]:
display(save(eda.schema_table(df[["flags", "instruction", "category", "intent", "response"]]).reset_index(names="column"), "eda_schema"))
display(save(eda.quality_report(df), "eda_quality"))
display(df.groupby("split").size().rename("rows").to_frame())

,column,dtype,non_null,null,empty_string,n_unique,example
0,flags,string,25545,0,0,370,BCIPZ
1,instruction,string,25545,0,0,25545,"I would like to acivate a card, can you help me?"
2,category,string,25545,0,0,9,CARD
3,intent,string,25545,0,0,26,activate_card
4,response,string,25545,0,0,25545,I'm here to assist you with that! Activating your card is an


,check,value
0,rows,25545
1,intents,26
2,categories,9
3,exact duplicate instructions,0
4,duplicate instructions (case/space-insensitive),110
5,"duplicate (instruction, response) pairs",0
6,duplicate responses,0
7,instructions mapped to >1 intent,0
8,responses containing {{placeholders}},14064
9,instructions containing {{placeholders}},0


,rows
split,
test,3831
train,17882
val,3832


## 2. Label distribution (intents, categories, language-variation flags)

In [3]:
counts = eda.plot_label_distribution(df)
display(counts.groupby("category")["n"].agg(["count", "sum"]).rename(columns={"count": "intents", "sum": "rows"}))
display(save(eda.plot_flags(df), "eda_flags"))

,intents,rows
category,,
ACCOUNT,3,2994
ATM,2,1983
CARD,6,5980
CONTACT,2,1997
FEES,1,947
FIND,2,1998
LOAN,6,5954
PASSWORD,2,1700
TRANSFER,2,1992


,flag,meaning,rows,share
0,B,basic syntax,25545,1.000000
1,C,coordinated (multi-part),20567,0.805128
2,I,interrogative,17954,0.702838
3,Z,errors & typos,13071,0.511685
4,L,semantic/lexical variation,12896,0.504835
5,Q,colloquial,12706,0.497397
6,P,polite,10960,0.429047
7,W,offensive,3729,0.145978
8,E,abbreviations,2728,0.106792
9,M,morphological variation,1794,0.070229


## 3. Lengths, vocabulary, n-grams, word cloud

In [4]:
try:  # subword lengths with the same tokenizer BERT will use
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(C.BERT_MODEL)
except Exception as e:
    print("tokenizer unavailable -> word counts only:", type(e).__name__); tok = None
lengths = eda.length_table(df, tok)
display(save(lengths.describe(percentiles=[.5, .9, .99]).T.round(1).reset_index(names="measure"), "eda_lengths"))
eda.plot_lengths(lengths)
eda.plot_response_length_by_intent(df)
if tok is not None:
    print(f"queries longer than 64 BERT tokens: {(lengths['instruction_subwords'] > 64).mean():.2%} -> max_length=64 is safe")

/Users/fangziming/Desktop/Uchicago课程/Multimodal/banking-support-agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (517 > 512). Running this sequence through the model will result in indexing errors


,measure,count,mean,std,min,50%,90%,99%,max
0,instruction_words,25545.0,11.081973,2.709542,2.0,11.0,14.0,18.56,23.0
1,response_words,25545.0,148.561441,59.103978,14.0,141.0,220.0,346.56,431.0
2,instruction_chars,25545.0,52.366334,14.978346,12.0,50.0,72.0,99.0,120.0
3,response_chars,25545.0,909.765277,382.208272,80.0,861.0,1366.0,2205.12,2808.0
4,instruction_subwords,25545.0,12.924839,3.352688,3.0,13.0,17.0,21.0,27.0
5,response_subwords,25545.0,195.375181,74.509214,23.0,190.0,283.0,438.56,540.0


queries longer than 64 BERT tokens: 0.00% -> max_length=64 is safe


In [5]:
vocab = pd.DataFrame([eda.vocab_stats(df["instruction"], "Bitext queries"), eda.vocab_stats(df["response"], "Bitext responses")])
display(save(vocab, "eda_vocab"))
uni, bi = eda.plot_top_ngrams(df["instruction"])
eda.plot_wordcloud(df["instruction"])
display(save(eda.distinctive_terms_per_intent(df), "eda_terms_per_intent"))

,corpus,documents,tokens,vocabulary,type_token_ratio,hapax_legomena,mean_words_per_doc
0,Bitext queries,25545,283089,3432,0.0121,1702,11.08
1,Bitext responses,25545,3795002,4460,0.0012,782,148.56


,intent,top_terms
0,activate_card,"activate, mobile, card, master, online, visa"
1,activate_card_international_usage,"usage, international, overseas, traveling, activate, abroad"
2,apply_for_loan,"loan, apply, applying, taking, looking, olan"
3,apply_for_mortgage,"apply, mortgage, applying, taking, looking, mortagge"
4,block_card,"lock, block, stolen, lost, mobile, blocking"
5,cancel_card,"cancel, mobile, card, master, visa, amex"
6,cancel_loan,"loan, cancel, pay, cancelling, canceling, dont"
7,cancel_mortgage,"mortgage, cancel, pay, canceling, cancelling, don"
8,cancel_transfer,"transfer, mistake, wrong, transferred, money, contact"
9,check_card_annual_fee,"annual, annuals, fees, fee, card, check"


## 4. Linguistic features: POS tags & named entities (spaCy, 3,000-row sample)

In [6]:
pos_df, ent_df = eda.pos_entity_profile(df["instruction"])
eda.plot_pos(pos_df)
display(save(pos_df, "eda_pos"), save(ent_df, "eda_entities_queries"))
_, ent_resp = eda.pos_entity_profile(df["response"], sample=1500)
display(save(ent_resp, "eda_entities_responses"))
# Interpretation hint: few real entities in queries (synthetic data has no names/amounts) vs. many
# placeholders in responses -> motivates the "no invented specifics" rule in our prompts.

,pos,count,share
0,VERB,7851,0.214140
1,PRON,7414,0.202220
2,NOUN,5726,0.156179
3,AUX,3080,0.084008
4,DET,2893,0.078908
5,PART,2540,0.069280
6,PUNCT,2379,0.064888
7,ADP,1686,0.045986
8,ADJ,1158,0.031585
9,SCONJ,961,0.026212


,entity,count,example
0,ORG,263,ATM
1,PERSON,95,Card
2,DATE,54,annual
3,WORK_OF_ART,3,a Master Card
4,NORP,2,byan
5,PRODUCT,2,Iwant


,entity,count,example
0,CARDINAL,6490,1
1,ORG,843,MasterCard
2,WORK_OF_ART,727,Register
3,PERSON,230,X.
4,DATE,217,the last 30 days
5,PRODUCT,60,Cancel
6,LAW,44,"the ""Manage Cards"
7,ORDINAL,34,First
8,EVENT,31,"the ""Forgot Password"
9,TIME,25,24-hour


## 5. Embedding map & intent similarity (sentence-transformers; TF-IDF fallback offline)

In [7]:
from src.utils import sample_per_group
sample = sample_per_group(df, "intent", 150)
emb_model = Embedder().fit(df.loc[df["split"] == "train", "instruction"].tolist())
print("embedding backend:", emb_model.backend)
emb = emb_model.encode(sample["instruction"].tolist())
eda.plot_embedding_map(emb, sample["category"].reset_index(drop=True))
sim_pairs = eda.plot_intent_similarity(emb, sample["intent"].reset_index(drop=True))
display(save(sim_pairs.head(10), "eda_most_similar_intent_pairs"))
# These pairs (e.g. cancel_loan vs cancel_mortgage) are where we expect classifier confusion -> check in Task 1.

W1122 22:03:41.772000 93296 .venv/lib/python3.11/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6634.01it/s]

embedding backend: sbert:sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/31 [00:00<?, ?it/s]

Batches:   3%|▎         | 1/31 [00:00<00:22,  1.32it/s]

... (14 outputs left out)

Batches: 100%|██████████| 31/31 [00:03<00:00, 10.21it/s]

Batches: 100%|██████████| 31/31 [00:03<00:00,  9.53it/s]

/Users/fangziming/Desktop/Uchicago课程/Multimodal/banking-support-agent/.venv/lib/python3.11/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/Users/fangziming/Desktop/Uchicago课程/Multimodal/banking-support-agent/.venv/lib/python3.11/site-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,intent_a,intent_b,cosine
318,get_password,set_up_password,0.923547
135,cancel_loan,cancel_mortgage,0.867857
234,check_loan_payments,check_mortgage_payments,0.846673
0,activate_card,activate_card_international_usage,0.812866
49,apply_for_loan,apply_for_mortgage,0.792316
81,apply_for_mortgage,check_mortgage_payments,0.784047
208,check_current_balance_on_card,check_recent_transactions,0.769614
58,apply_for_loan,check_loan_payments,0.757436
52,apply_for_loan,cancel_loan,0.746230
75,apply_for_mortgage,cancel_mortgage,0.745713


## 6. Synthetic-data checks: diversity, placeholders, anomalies, synthetic vs real

In [8]:
display(save(eda.template_diversity(df), "eda_template_diversity"))
display(save(eda.placeholder_table(df), "eda_placeholders"))
for name, t in eda.anomalies(df).items():
    print(f"\n### {name} ({len(t)})"); display(t)

,field,distinct_texts_%,distinct_opening_5grams_%,most_common_opening,rows_with_that_opening_%,most_common_closing,rows_with_that_closing_%
0,instruction,100.0,25.21,i got to cancel a,1.04,could i get some help,2.77
1,response,100.0,4.60,i'm here to assist you,18.27,know i'm here to help,10.10


,placeholder,occurrences
0,Customer Support Phone Number,7879
1,Company Website URL,6063
2,Credit Card,5310
3,Customer Support Working Hours,3024
4,Email Address,1100
5,Customer Support Team,851
6,Customer Support Email Address,850
7,Name,733
8,Password,573
9,Customer Service Phone Number,564



### shortest_queries (5)


,row_id,intent,flags,instruction,response
18112,bx_18112,dispute_ATM_withdrawal,BZ,disputea withdrawal,"It's our responsibility, and we're sorry for any inconvenience caused by the withdrawal that you would like to dispu..."
21009,bx_21009,get_password,BZ,geta password,Definitely! I'm here to assist you in retrieving your password so you can regain access to your account. Let's get s...
2516,bx_02516,apply_for_loan,BLZ,takeout a loan,"I'm here to help you with that! Taking out a loan is an important decision, and I'll guide you through the process. ..."
4648,bx_04648,block_card,BL,block a Visa,I'm sorry to hear that you need to block your {{Credit Card}}. I'm here to assist you with that. To ensure the secur...
6208,bx_06208,cancel_loan,BZ,cancel an loan,I comprehend that you're seeking assistance with canceling a loan. I'm here to help guide you through the process. H...



### longest_queries (5)


,row_id,intent,flags,instruction,response
8131,bx_08131,cancel_transfer,BCEI,"I have made a mistake and I have transferred money to the wrong account, can you help me to cancel a bank transfer?",I'm sorry to hear that you've made a mistake with your bank transfer. I'll do my best to assist you in canceling it....
8460,bx_08460,cancel_transfer,BCEIP,"I have made a mistake and I have transferred money to the wrong account, would it be possible to cancel a bank trans...",Our team regrets and is sorry for the inconvenience you've faced with your bank transfer. I understand that making a...
8037,bx_08037,cancel_transfer,BCE,"I have made a mistake and I have transferred money to the wrong account, I need assistance to cancel a bank transfer","We acknowledge the issue, and we're sorry for any inconvenience caused by transferring money to the wrong account. I..."
8130,bx_08130,cancel_transfer,BCEIPZ,"I made a mistake and I have transferred money tgo the wrong account, would it be possible to cancel a bank transfer?",I'm really sorry to hear about the mistake you made with the bank transfer. I'll do my best to assist you in canceli...
8171,bx_08171,cancel_transfer,BCEIZ,"I have made a mistake and I have trnasferred money to the wrong account, can you help me to cancel a transfer?","I understand the urgency to rectify the situation, and I'm here to assist you in canceling the transfer. I apologize..."


... (4 outputs left out)


### typo_flag_Z (5)


,row_id,intent,flags,instruction,response
0,bx_00000,activate_card,BCIPZ,"I would like to acivate a card, can you help me?",I'm here to assist you with that! Activating your card is an important step to starting and enjoying its benefits. H...
1,bx_00001,activate_card,BCILZ,"I have to activate an Visa online, how can I do it?","I'm here to assist you with activating your {{Credit Card}} online. To activate your {{Credit Card}}, please follow ..."
2,bx_00002,activate_card,BCIPQZ,I'd like to actiate a card where do i do it,I can help you with that! Activating your card is easy. Here's what you need to do:\n\n1. If you received your card ...
3,bx_00003,activate_card,BCLPQZ,I'd likke to activate a visa on mobile i need assistance,I'm here to assist you with activating your {{Credit Card}} on your mobile device. Activating your card is a simple ...
4,bx_00004,activate_card,BCILPZ,"I would ilke to activate a credit card online, will you help me?",I'm here to assist you with activating your credit card online. Activating your credit card is a simple and convenie...



### unbalanced_placeholder_braces (5)


,row_id,intent,flags,instruction,response
41,bx_00041,activate_card,BCIQZ,wanna activate an amexx online how do i do it,I'm here to guide you through the process of activating your American Express (Amex) card online. Let's get started:...
109,bx_00109,activate_card,BCIPZ,"I got to activate an Amex on mobile, wil you help me?","Yes, of course! I'm here to help you activate your {{Credit Card}} on your mobile device. Follow these simple steps:..."
152,bx_00152,activate_card,BCIQ,i got to activate an amex where can i do it,"I'm here to help you activate your {{Credit Card}}. To activate your {{Credit Card}}, you have a few options:\n\n1. ..."
199,bx_00199,activate_card,BQZ,help me to activate an amex on mobvile,I'm here to assist you with activating your American Express (Amex) card on your mobile device. Here's what you need...
262,bx_00262,activate_card,BCIP,"I got to activate an Amex on mobile, will you help me?",I'm here to assist you with activating your American Express card on your mobile device. Here's what you need to do:...


In [9]:
display(save(eda.synthetic_vs_real(df, b77), "eda_synthetic_vs_real"))
# Real customers write differently (length, vocabulary) -> justification for the BANKING77 external test.

,corpus,documents,tokens,vocabulary,type_token_ratio,hapax_legomena,mean_words_per_doc
0,Bitext queries,25545,283089,3432,0.0121,1702,11.08
1,BANKING77 queries,13083,153863,2602,0.0169,986,11.76


## 7. Findings to carry into the report (filled in from the run of 2026-11-22)
1. **Class balance:** the 26 intents are nearly balanced (947-1,000 rows each), except `get_password` (799)
   and `set_up_password` (901). CARD and LOAN hold 6 intents each. → Accuracy and Macro-F1 will be close; we report both.
2. **Templating:** queries are short (median 11 words, max 23; 0% exceed 64 BERT tokens) and heavily templated:
   only 25% of queries and 4.6% of responses have a distinct opening 5-gram, and 18% of responses start with
   "I'm here to assist you". Even after the grouped split, 52% of test queries have a near-paraphrase
   (char cosine ≥ 0.9, same intent) in train (random split: 60%); exact duplicates: 0.
   → In-distribution scores will be near the ceiling and reference-similarity metrics are inflated:
   use the rubric + LLM judge, and BANKING77 for generalisation.
3. **Placeholders:** 55% of responses (14,064) contain `{{placeholders}}` and no query does. Most common:
   Customer Support Phone Number (7,879), Company Website URL (6,063), Credit Card (5,310), Working Hours (3,024).
   3 responses are refusals ("I'm unable to assist…") and sit in the train split / knowledge base.
   → Policy rule 2 (placeholders instead of invented facts) and the `invented_*` checks.
4. **Most similar intents** (mean embedding cosine): get_password vs set_up_password (0.92), cancel_loan vs
   cancel_mortgage (0.87), check_loan_payments vs check_mortgage_payments (0.85), activate_card vs
   activate_card_international_usage (0.81). → Expected confusions; why the agent filters retrieval by intent
   only above a confidence threshold.
5. **Synthetic vs real gap:** BANKING77 queries have a similar length (11.8 vs 11.1 words) but only 10% of their
   word types appear in Bitext queries (69% of tokens). Frequent real words absent from Bitext: why, pending,
   declined, charged, refund, exchange, app, pin. Real customers often ask *why* something happened to a
   specific transaction; Bitext only asks *how to* do a task. Only 13 of 77 BANKING77 intents map to Bitext
   with high/medium confidence; 61 are unsupported. → External test (`b77_real_inscope`) and escalation test
   (`b77_unsupported`).